# Milestone 1 crawler on Colab (CPU)

One machine crawls the **whole** set (`id_ranges.csv`, D8). Put `S2_API_KEY` in Colab **Secrets** (never in the notebook).

**Chạy khi rảnh, dừng khi bận — không mất dữ liệu** if you follow this loop every session:

1. Setup (clone, mount Drive, secrets) — once per session
2. **Restore** Drive → `/content/data` — always before crawl
3. **Crawl** — same command every time; manifest skips finished IDs (auto-backup every 1000 IDs to Drive when env is set)
4. When you stop: **Runtime → Interrupt**, then run **Backup** if you want a final rsync
5. Next session: repeat from step 1

After **disconnect**, re-run setup from the top. If Drive mount fails (`Mountpoint must not already contain files`), use **Runtime → Restart session**, then run again (do **not** use `force_remount` on a non-empty `/content/drive`).

Data lives on Colab local disk while crawling (SQLite + Drive FUSE is unsafe). Backup copies the full `data/` folder to `MyDrive/arxiv-data/`.

In [ ]:
import os

if not os.path.isdir("/content/arxiv-pipeline"):
    !git clone https://github.com/GiaaPhuc/arxiv-pipeline.git
else:
    !git -C /content/arxiv-pipeline pull --ff-only
%cd /content/arxiv-pipeline/src
!pip install -q -r requirements.txt

In [ ]:
import os
import shutil
import sqlite3
import subprocess
import threading
import time

from google.colab import drive, userdata

_DRIVE_MOUNT = "/content/drive"


def _drive_usable() -> bool:
    my_drive = os.path.join(_DRIVE_MOUNT, "MyDrive")
    if not os.path.isdir(my_drive):
        return False
    try:
        os.listdir(my_drive)
        return True
    except OSError:
        return False


if _drive_usable():
    print("Google Drive already mounted.")
else:
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if os.path.isdir(_DRIVE_MOUNT) and os.listdir(_DRIVE_MOUNT):
        raise RuntimeError(
            "Cannot mount Drive: /content/drive still has files "
            "(common after disconnect). Runtime → Restart session, "
            "then run this notebook from the top."
        )
    drive.mount(_DRIVE_MOUNT)

os.environ["S2_API_KEY"] = userdata.get("S2_API_KEY")
os.environ["CRAWLER_CONTACT"] = "your-email@example.com"  # User-Agent contact
os.environ["CRAWLER_DATA_DIR"] = "/content/data"
os.environ["CRAWLER_ARXIV_MIN_INTERVAL"] = "3.5"  # Colab: small margin above 3 s

_TAR_DRIVE = "/content/drive/MyDrive/arxiv-data.tar"
_bk_lock = threading.Lock()
_auto_started = False


def backup_now():
    """Consistent manifest snapshot + data/ -> ONE tar on Drive (atomic replace)."""
    with _bk_lock:  # manual + auto backup must not share the temp tar concurrently
        os.makedirs("/content/_bk", exist_ok=True)
        src = "/content/data/manifest.sqlite"
        if os.path.isfile(src):
            s = sqlite3.connect(f"file:{src}?mode=ro", uri=True)
            d = sqlite3.connect("/content/_bk/manifest.sqlite")
            s.backup(d)
            d.close()
            s.close()
        # tar excludes are global, so add the snapshot in a second step.
        r = subprocess.run(
            [
                "tar",
                "-cf",
                "/content/arxiv-data.tar",
                "--exclude=.tmp",
                "--exclude=manifest.sqlite*",
                "-C",
                "/content/data",
                ".",
            ]
        )
        if r.returncode not in (0, 1):  # 1 = "file changed as we read it" (crawl is live)
            raise RuntimeError(f"tar failed: {r.returncode}")
        subprocess.run(
            ["tar", "-rf", "/content/arxiv-data.tar", "-C", "/content/_bk", "manifest.sqlite"],
            check=True,
        )
        shutil.copyfile("/content/arxiv-data.tar", _TAR_DRIVE + ".part")
        os.replace(_TAR_DRIVE + ".part", _TAR_DRIVE)
        print("backup done", time.strftime("%H:%M:%S"))


def start_auto_backup(every_s=1800):
    global _auto_started
    if _auto_started:
        return
    _auto_started = True

    def loop():
        while True:
            time.sleep(every_s)
            try:
                backup_now()
            except Exception as e:
                print("auto backup failed:", e)

    threading.Thread(target=loop, daemon=True).start()

In [ ]:
# Plan only: no network.
!python run_crawl.py --ranges id_ranges.csv --dry-run

In [ ]:
# Restore from Drive - run EVERY new session (safe on first run).
# Preferred: one big tar = one Drive transfer (far faster than many small files).
# Fallback: legacy folder arxiv-data/ (slow rsync, one time), then convert it to a tar.
!mkdir -p /content/data
!if [ -f /content/drive/MyDrive/arxiv-data.tar ]; then \
  cp /content/drive/MyDrive/arxiv-data.tar /content/arxiv-data.tar && \
  tar -xf /content/arxiv-data.tar -C /content/data && rm /content/arxiv-data.tar; \
  elif [ -d /content/drive/MyDrive/arxiv-data ]; then \
  echo 'Legacy folder backup found: slow one-time rsync...' && \
  rsync -a /content/drive/MyDrive/arxiv-data/ /content/data/ && touch /content/.migrate_to_tar; \
  else echo 'No backup on Drive (first run).'; fi
if os.path.exists("/content/.migrate_to_tar"):
    backup_now()  # convert legacy folder to tar so next restore is fast
    os.remove("/content/.migrate_to_tar")
start_auto_backup(1800)  # tar backup to Drive every 30 min while crawling

In [ ]:
# Crawl: re-run the SAME active line after every restore to resume.
# Exit code 3 = arXiv still rate-limiting after waits — wait 30–60 min, Backup, try again later.
# First session — pilot 30 IDs (switch to full crawl after pilot OK):
!python run_crawl.py --ranges id_ranges.csv --limit 30
# !python run_crawl.py --ranges id_ranges.csv

### Dừng an toàn (trước khi đóng tab / tắt máy / hết giờ Colab)

1. **Runtime → Interrupt execution** (or wait until you want to stop — Ctrl+C is OK; manifest keeps progress).
2. Run the **Backup** cell below and wait until `backup done` is printed.
3. You can disconnect; next session: **Restore** then the same crawl command.

Do **not** re-run **Restore** in the same session after crawling unless you already backed up — it would overwrite local data with the last Drive snapshot.

In [ ]:
# Backup - run after Interrupt and before closing the session.
backup_now()

In [ ]:
# ONLY when crawl finished (no exit 3 / no pending pairs) — not after a partial run:
# !python validate_corpus.py --data-dir $CRAWLER_DATA_DIR
# !python export_data.py --data-dir $CRAWLER_DATA_DIR